# Programmation Python  pour les mathématiques

**Julien Guillod**, [Sorbonne Université](http://www.sorbonne-universite.fr/),
Licence <a href="https://creativecommons.org/licenses/by-nc-nd/4.0/">CC BY-NC-ND</a>

L'entier des chapitres est disponible au format
[HTML](https://python.guillod.org/) et [PDF](https://python.guillod.org/python.pdf).
Ce notebook peut également être exécuté sur [GESIS](https://notebooks.gesis.org/binder/v2/gh/guillod/python/master?urlpath=lab/tree/chap06.ipynb).

# 6 Algèbre

<div id="ch:algebre"></div>

Tout d'abord une méthode de résolution de système linéaire par un algorithme direct est étudiée, puis une méthode itérative sera utilisée pour déterminer le vecteur propre associé à la plus grande valeur propre d'une matrice. Enfin les groupes générés par un ensemble de permutations seront étudiés.

**Concepts abordés:**

* méthode de résolution directe (décomposition LU)

* algorithme *in place*

* méthode itérative (puissance itérée)

* groupes de permutations

* orbite et stabilisateur

# Exercice 6.1: Décomposition LU

<div id="exer:algebre-LU"></div>

La décomposition LU consiste à décomposer une matrice $A$ de taille $n \times n$ sous la forme $A=LU$ où $L$ est une matrice triangulaire inférieure avec des 1 sur la diagonale et $U$ une matrice triangulaire supérieure. Une fois la décomposition $A=LU$ d'une matrice connue, il est alors très facile de résoudre le problème linéaire $A\boldsymbol{x} = \boldsymbol{b}$ en résolvant d'abord $L\boldsymbol{y} = \boldsymbol{b}$ puis $U\boldsymbol{x} = \boldsymbol{y}$. L'avantage de la décomposition LU sur l'algorithme de Gauss, par exemple, est qu'une fois la décomposition LU connue, il est possible de résoudre le système linéaire rapidement avec des seconds membres différents.

Vu que $l_{ik}=0$ si $k>i$, nous avons:

$$
a_{ij} = \sum_{k=1}^n l_{ik}u_{kj} = l_{ii}u_{ij} + \sum_{k=1}^{i-1} l_{ik}u_{kj} \,,
$$

et donc comme $l_{ii}=1$:

$$
u_{ij} = a_{ij} - \sum_{k=1}^{i-1} l_{ik}u_{kj} \,.
$$

D'un autre côté, vu que $u_{kj}=0$ si $k>j$, alors:

$$
a_{ij} = \sum_{k=1}^n l_{ik}u_{kj} = l_{ij}u_{jj} + \sum_{k=1}^{j-1} l_{ik}u_{kj} \,,
$$

et donc si $u_{jj}\neq0$:

$$
l_{ij} = \frac{1}{u_{jj}} \left( a_{ij} - \sum_{k=1}^{j-1} l_{ik}u_{kj} \right) \,.
$$

Ainsi, si les $(i-1)$ premières lignes de $U$ et les $(i-1)$ premières colonnes de $L$ sont connues, il est possible de déterminer la $i$-ème ligne de $U$ par:

$$
u_{ij} = a_{ij} - \sum_{k=1}^{i-1} l_{ik}u_{kj} \,, \quad j \geq i \,,
$$

puis la $i$-ème colonne de $L$ par:

$$
l_{ji} = \frac{1}{u_{ii}} \left( a_{ji} - \sum_{k=1}^{i-1} l_{jk}u_{ki} \right) \,, \quad j > i\  \,.
$$

Pour que la décomposition LU d'une matrice $A$ soit possible il faut que les $u_{ii}$ ne soient jamais nuls. Cela est effectivement le cas lorsque la matrice $A$ et toutes ses sous-matrices principales sont inversibles.

**a)**
Écrire une fonction `LU(A)` qui retourne le résultat de la décomposition LU d'une matrice.

In [21]:
import numpy as np

def LU(A):
    """Décompose une matrice sur numpy en sa décomposition LU, avec L en premier output et U en deuxième, avec les descriptions de L et U dans l'énoncé ci-dessus

    Args:
        A (numpy.ndarray): La matrice à transformer
    """
    # On initialise nos matrices L et U , que l'on modifiera par la suite
    n = len(A)
    U = np.full((n,n),0)
    L = np.identity(n)
    
    for i in range(n):
        l = L[i,:]
        for j in range(i,n):
            u = U[:,j]
            U[i][j]=A[i][j]-np.dot(l,u)
        if U[i,i]==0:
            return "Decomposition Impossible"
        for j in range(i+1,n):
            L[j,i]= (A[j,i]-np.dot(L[j,:],U[:,i]))/U[i][i]
    return L,U
    

In [28]:
L = np.array([[1,2,3,4],[3,4,5,4]])
L[1,:2]

array([3, 4])

**b)**
Donnée la décomposition LU d'une matrice $A$, écrire une fonction `solve(L,U,b)` qui résout le système linéaire $A\boldsymbol{x} =  \boldsymbol{b}$.

In [31]:
def solve(L,U,b):
    # On resout Ly = b
    y = np.zeros(len(b))
    for i in range(len(y)):
        y[i]=b[i]- L[i]@y
    
    # On resout Ux = y
    x = np.zeros(len(b))
    for i in range(1,len(x)+1):
        x[-i]=(y[-i]-U[-i]@x)/U[-i][-i]
    return x

**c)**
Écrire une fonction `LU_inplace(A)` qui ne crée pas de nouveaux tableaux pour $L$ et $U$ mais modifie $A$ de sorte que sa partie triangulaire inférieure (sans la diagonale) corresponde à $L$ et sa partie triangulaire supérieure (avec la diagonale) corresponde à $U$. Faire également une version `solve_inplace` prenant en entrée la sortie de `LU_inplace` et retournant la solution $\boldsymbol{x}$, sans utiliser de nouveaux tableaux.

In [ ]:
def LU_inplace(A):
    L,U = LU(A)
    for i in range(A):
        A[i]=np.concatenate(L[i,:i], U[i])
    return A

def solve_inplace(A,b):
    L,U = LU(A)
    return solve(L,U,b)

**d)**
En utilisant la décomposition LU de la matrice $A$, écrire une fonction `det(A)` qui retourne son déterminant.

In [34]:
def det(A):
    U = LU(A)[1]
    det = 1
    for i in range(len (U)):
        det*=U[i,i]
    return det

# Exercice 6.2: Méthode de la puissance itérée

<div id="exer:algebre-power"></div>

Le but de cet exercice est de déterminer le vecteur propre d'une matrice associé à la valeur propre de plus grand module, en supposant que celle-ci est unique. Étant donné une matrice réelle $A$ de taille $n\times n$ et un vecteur $\boldsymbol{x}_0\in\mathbb{R}^n$, la suite de vecteurs $(\boldsymbol{x}_k)_ {k\in\mathbb{N}}$ est définie par:

$$
\boldsymbol{x}_{k+1} = \frac{A\boldsymbol{x}_k}{\Vert A\boldsymbol{x}_k\Vert} \,.
$$

En supposant par exemple que la matrice $A$ soit diagonalisable, alors il est possible de montrer que la suite $(\boldsymbol{x}_k)_ {k\in\mathbb{N}}$ converge à un signe près vers le vecteur propre associé à la plus grande valeur propre de $A$ en valeur absolue.

**Indication:**
La première étape est de remarquer que:

$$
\boldsymbol{x}_k = \frac{A^2 \boldsymbol{x}_{k-2}}{\Vert A^2 \boldsymbol{x}_{k-2}\Vert} = \dots = \frac{A^k \boldsymbol{x}_0}{\Vert A^k \boldsymbol{x}_0\Vert} \,.
$$

Vu que $A$ est diagonalisable, soit ($\boldsymbol{v}_1, \boldsymbol{v}_2, \dots, \boldsymbol{v}_n)$ une base de vecteurs propres de $A$ associés aux valeurs propres $\lambda_1, \lambda_2, \dots, \lambda_n$. Sans perte de généralité, on suppose que $\lambda_1$ est la valeur propre de plus grand module, c'est-à-dire $|\lambda_1|>\max(|\lambda_2|,|\lambda_3|,\dots,|\lambda_n|)$. À noter que cela implique que $\lambda_1$ est réelle. Le vecteur $\boldsymbol{x}_0$ se décompose dans la base:

$$
\boldsymbol{x}_0 = \sum_{i=1}^n c_i \boldsymbol{v}_i \,,
$$

ainsi en supposant que $c_1\neq0$:

$$
A^k \boldsymbol{x}_0 = \sum_{i=1}^n c_i \lambda_i^k \boldsymbol{v}_i
= c_1 \lambda_1^k \left(\boldsymbol{v}_1 + \sum_{i=2}^n \frac{c_i}{c_1} \left(\frac{\lambda_i}{\lambda_1}\right)^k \boldsymbol{v}_i \right) .
$$

Vu que $|\lambda_1| > |\lambda_i|$ pour $i\geq2$, alors:

$$
\lim_{k\to\infty}\left(\boldsymbol{v}_1 + \sum_{i=2}^n \frac{c_i}{c_1} \left(\frac{\lambda_i}{\lambda_1}\right)^k \boldsymbol{v}_i \right) = \boldsymbol{v}_1 \,,
$$

puisque $\left|\frac{\lambda_i}{\lambda_1}\right| < 1$.
Par conséquent,

$$
\lim_{k\to\infty}\operatorname{sign}{(\lambda_1)}^k \boldsymbol{x}_k = \lim_{k\to\infty} \left(\frac{\lambda_1}{|\lambda_1|}\right)^k \frac{A^k \boldsymbol{x}_0}{\Vert A^k \boldsymbol{x}_0\Vert} = \operatorname{sign}{c_1} \frac{\boldsymbol{v}_1}{\Vert\boldsymbol{v}_1\Vert} \,.
$$

En choisissant $\boldsymbol{x}_0$ aléatoirement, alors avec probabilité un $c_1\neq0$ et donc la suite $(\boldsymbol{x}_k)_ {k\in\mathbb{N}}$ converge à un signe près vers un vecteur propre normalisé associé à la valeur propre de plus grand module.

**a)**
Définir une fonction `puissance(A, x0, k)` qui retourne $\boldsymbol{x}_k$. Avec cette fonction, déterminer le plus grand vecteur propre de la matrice:

$$
A = \begin{pmatrix}0.5 & 0.5\\ 
0.2 & 0.8
\end{pmatrix} \,.
$$

**Réponse:**
Le plus grand vecteur propre est $\pm(0.70710678, 0.70710678)$.

In [3]:
import numpy as np

def puissance(A,x0,k):
    """Retourne x_k comme décrit plus tot

    Args:
        A (np.ndarray): Matrice diagonalisable
        x0 (np.ndarray): Vecteur de départ
        k (int): Nombre d'itérations
    """
    
    A_k = np.linalg.matrix_power(A,k)
    x_k = np.dot(A_k,x0)/np.linalg.norm(np.dot(A_k,x0)) 
    
    return x_k

In [4]:
# On teste notre code
# Note : On pourrait très bien prendre x0 aléatoirement via np.random.rand()
A = np.array([[0.5, 0.5], [0.2, 0.8]])
x0 = np.array([1,3])
k = 200

puissance(A,x0,k)

array([0.70710678, 0.70710678])

**b)**
Déterminer la valeur propre associée au vecteur propre précédent.

**Indication:**
Si $\boldsymbol{v}$ est un vecteur propre normalisé de $A$, alors la valeur propre associée est donnée par $\lambda = A\boldsymbol{v}\cdot\boldsymbol{v}$.

In [5]:
x = puissance(A,x0,k)
Ax = A @ x
eigenvalue = Ax @ x
print("La valeur propre associée au vecteur annoncé plus tot est ",eigenvalue)

La valeur propre associée au vecteur annoncé plus tot est  1.0


**c)**
Écrire un programme permettant de calculer automatiquement la valeur propre de plus grand module et le vecteur propre associé d'une matrice carrée avec une certaine précision donnée.

In [6]:
def question_c(A,precision = 10**(-7),max_iterations=800):
    """Pour une matrice A, une niveau de précision donnée et un maximum d'itéarions, retourne dans l'ordre (si convergence):
    La Valeur propre obtenue par cette méthode, son vecteur propre associé et le nombre d'itérations nécessaires pour sa convergence.

    Args:
        A (np.ndarray): Matrice étudiée
        precision (Float, optional): Niveau de précision attendu. Defaults to 10**(-6).
        max_iterations (int, optional): Nombre maximum d'itértions. Defaults to 800.
        
    Outputs :
        eigenvalue (float) : Valeur propre obtenue par l'algorithme
        eigenvector (np.ndarray) : Vecteur propre correspondant à la valeur propre obtenue
        i (int) : nombre d'itérations
        success? (boulean): Les résultats obtenus respectent-ils le niveau de précision exigé ?
    """
    v = np.random.random(len(A))
    
    
    for i in range(1, max_iterations):
        Av = A @ v
        eigenvalue = Av @ v
        if np.linalg.norm(Av - eigenvalue*v) < precision:
            v = Av/np.linalg.norm(Av)
            return(eigenvalue,v,i,True)
        
        v = Av/np.linalg.norm(Av)
        
    return (eigenvalue,v,i,False)            

In [7]:
# On teste notre code
A = np.array([[0.5, 0.5], [0.2, 0.8]])
question_c(A,max_iterations=1000)


(0.9999999709529206, array([0.7071068 , 0.70710676]), 14, True)

**d)**
Regarder la documentation de Numpy pour trouver les fonctions permettant de calculer les vecteurs propres et valeurs propres d'une matrice.

In [8]:
A = np.array([[0.5, 0.5], [0.2, 0.8]])

np.linalg.eig(A)

(array([0.3, 1. ]),
 array([[-0.92847669, -0.70710678],
        [ 0.37139068, -0.70710678]]))

**e)**
Comparer la rapidité du code précédent et des fonctions Numpy pour des matrices de tailles $n\times n$ avec $n=10,100,1 000$.

**Indication:**
En prenant par exemple des matrices dont les coefficients sont générés aléatoirement dans l'intervalle $(0,1)$, le théorème de Perron-Frobenius assure l'existence d'une unique valeur propre de module maximal, et celle-ci est positive.

In [9]:
A = np.random.random((10,10))
B = np.random.random((100,100))
C = np.random.random((1000,1000))

In [10]:
%%timeit
# On compare pour un niveau de précision de 10**(-7)

question_c(A,max_iterations=1000)
question_c(B,max_iterations=1000)
question_c(C,max_iterations=1000)

The slowest run took 5.49 times longer than the fastest. This could mean that an intermediate result is being cached.
3.95 ms ± 2.83 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [11]:
%%timeit

np.linalg.eig(A)
np.linalg.eig(B)
np.linalg.eig(C)


1.17 s ± 125 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Il semble que fournir une approximation permette d'obtenir des résultats plus rapidement que via la fonction numpy, laquelle, certainement mieux codée, est ralentie par une attente d'un résultat infiniment plus précis.

# Exercice 6.3: Groupes de permutations

<div id="exer:algebre-permutations"></div>

Le but est d'étudier les groupes de permutations en développant des algorithmes pour les caractériser. Un groupe de permutations $G \subset S_n$ est défini comme étant généré par un certain nombre de permutations: $G = \langle g_1, g_2,\dots,g_k \rangle$, avec $g_i\in S_n$ une permutation de l'ensemble $\{1,2,\dots,n\}$. Une permutation

$$
g = \begin{pmatrix}1 & 2 & 3 & 4\\ 
4 & 1 & 2 & 3
\end{pmatrix} \,,
$$

peut être représentée en python par le tuple `g = (0, 4, 1, 2, 3)`. Le zéro est ajouté afin de se conformer avec l'indexation à partir de zéro de Python et ainsi de simplifier un peu les implémentations. Cela veut simplement dire que le sommet `0` va sur le sommet `0`. À noter que cet exercice se prête particulièrement bien à une implémentation orientée objet, et dans ce cas les questions peuvent être adaptées en conséquence.

**a)**
Écrire une fonction `product(g1,g2)` qui calcule le produit de deux permutations.

In [1]:
def product(g1,g2):
    g = [0]
    for  i in range(1,len(g1)):
        g.append(g1[g2[i]])
    return tuple(g)

In [2]:
g = (0,4,1,2,3)
h = (0,2,4,3,1)
product(g,h)

(0, 1, 3, 2, 4)

**b)**
Écrire une fonction `inverse(g)` qui calcule l'inverse d'une permutation.

In [3]:
def inverse(g):
    inv=list(g) # ici sert uniquement à créer un tuple de même longueur
    for i in range(len(g)):
        inv[g[i]]=i
    return tuple(inv)

g = (0,4,1,2,3)
h = inverse(g)

product(g,h)

(0, 1, 2, 3, 4)

**c)**
Écrire une fonction qui retourne la décomposition d'une permutation sous forme de cycles représentés par une liste du tuples.

In [4]:
def decomposition(g):
    g1 = list(g)
    decomp = []
    visited =[]
    for i in range(1,len(g1)):
        if  i not in visited:
            c = i
            cycle = [i]
            visited.append(i)
            while g1[c]!=i: # On parcourt le cycle auquel appartient i
                c = g1[c]
                cycle.append(c)
                visited.append(c)
            decomp.append(tuple(cycle)) 
            #On ajoute le cycle trouvé dans la liste des décompositions
    return decomp

In [5]:

#On teste notre code
g = (0,4,8,2,3,7,5,6,1)
decomposition(g)

[(1, 4, 3, 2, 8), (5, 7, 6)]

**d)**
Écrire une fonction qui retourne la permutation correspondant à une liste de cycles, c'est-à-dire qui fait l'inverse de la fonction précédente.

In [19]:
def permut(cycles):
    # Ordre de la permutation
    n = 0
    for c in cycles:
        n+=(len(c)) # Le (-1) corrige la présence de 0 en début de chaque cycle
    #On initialise notre permutation
    p = [0]*(n+1)
    
    for c in cycles :
        p[c[-1]]=c[0]
        for i in range(len(c)-1):
            p[c[i]]=c[i+1]
    return p
    
        

### On teste notre code

In [18]:
g = (0,4,8,2,3,7,5,6,1)
h = decomposition(g)


[(1, 4, 3, 2, 8), (5, 7, 6)]

In [20]:
permut(h)

[0, 4, 8, 2, 3, 7, 5, 6, 1]

**e)**
<span style="color:red">!</span> En python un groupe $G = \langle g_1,g_2,\dots,g_k \rangle$ engendré par une famille de permutations, peut être représenté par une liste de permutations `G = [g1,g2,...,gk]`. Écrire une fonction `orbit(G,x)` qui retourne l'orbite d'un point $x\in\{1,2,\dots,n\}$:

$$
O_x = Gx = \{gx, g \in G\} \,.
$$

**Indication:**
Ne pas calculer l'ensemble des éléments du groupe, cela fait une liste beaucoup trop longue. Construire la liste $(X^0,X^1,\dots,X^N)$ d'ensembles disjoints définie récursivement par $X^0 = \{x\}$ et $X^n$ comme l'ensemble des éléments nouveaux de la forme $g_i y$ avec $1 \leq i \leq k$ et $y\in X^{n-1}$:

$$
X^{n}=\left(\bigcup_{i=1}^{k}g_{i}X^{n-1}\right)\setminus\left(\bigcup_{i=1}^{n-1}X^{i}\right) \,.
$$


**f)**
<span style="color:red">!!</span> Définir une fonction `stabilizer(G,x)` qui retourne le stabilisateur d'un point $x\in\{1,2,\dots,n\}$:

$$
G_x = \{g \in G : g x = x \} \,,
$$

sous la forme d'un ensemble de générateurs.

**Indication:**
Utiliser le théorème ou lemme de Schreier.